In [ ]:
# Registro de ejecucion: fecha, fuentes, licencias y versiones de los modelos
# (lo exige el enunciado como primera celda)
import datetime, platform
from importlib.metadata import version, PackageNotFoundError

import pandas as pd

print("Fecha de ejecucion:", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python:", platform.python_version(), "·", platform.system(), platform.release())
for paquete in ["paddleocr", "faster-whisper", "torch", "pandas", "plotly"]:
    try:
        print(f"{paquete}: {version(paquete)}")
    except PackageNotFoundError:
        print(f"{paquete}: no instalado")

pd.read_csv("datos/fuentes.csv")

# 01 — Extraccion: OCR, transcripcion, limpieza y EDA

## Decisiones sobre el corpus de documentos

### Decision 1: 8 paginas impresas, elegidas por dificultad

Los tres PDF impresos dan 54 paginas (`extraer_paginas.py`, 300 ppp), todas escaneo puro,
sin capa de texto. El minimo del enunciado es 8 impresos, dentro de 25 documentos en total.
Se usan **8 paginas**, elegidas **antes de correr el OCR** para que varien en dificultad:
portada, texto corrido y una pagina con tablas de cifras.

| PDF | Pagina | Contenido | Por que entra |
|---|---|---|---|
| Carta a Gamarra, Cuzco 1832 | p001 | portada | tipografia decorativa, papel amarillento |
| Carta a Gamarra, Cuzco 1832 | p003 | inicio de la carta | texto corrido denso, tipografia antigua |
| Carta a Gamarra, Cuzco 1832 | p023 | Notas | la pagina mas manchada; el folleto no tiene tablas |
| Memoria BCRP 1922 | p001 | portada y directorio | cifras de capital y nombres propios |
| Memoria BCRP 1922 | p002 | texto corrido | montos en libras peruanas dentro del texto |
| Memoria BCRP 1922 | p005 | balance al 31-12-1922 | tabla de cifras en letra pequena |
| Memoria BCRP 1923 | p003 | texto corrido | montos dentro del texto, escaneo gris |
| Memoria BCRP 1923 | p014 | Cuadro N.º 3 | tabla densa de cifras mensuales |

**Paginas descartadas (46)**, movidas a `datos/raw/documentos/paginas/_descartadas/`.
No son ilegibles: quedan fuera para ajustar el corpus al minimo y mantener el OCR manejable en CPU.

- Carta a Gamarra: p002, p004–p022, p024, p025 (22 paginas)
- Memoria BCRP 1922: p003, p004, p006, p007 (4 paginas)
- Memoria BCRP 1923: p001, p002, p004–p013, p015–p022 (20 paginas)

Las Memorias de 1924 a 1929 no entran como impresos: traen capa de texto (corrompida) y se usan
solo como hallazgo de partida.

### Decision 2: la foto de la boleta se usa sin tapar nada

`Foto_boletas.jpeg` se reviso a mano antes de incluirla. No contiene datos de una persona real:

- Cliente: "CLIENTE EVENTUAL", con DNI `00000001` (valor generico de boleta sin cliente identificado).
- RUC, direccion y telefono pertenecen a la empresa emisora, no a una persona natural.
- "Vendedor TULLUMAYO1" es un codigo de caja, no un nombre.

Si alguna foto nueva trae un nombre o DNI real, se tapa en la imagen antes de incluirla:
seudonimizar solo el texto del OCR no basta si la imagen se publica.